# CIOdD-UCR PDF Intelligence Pipeline V5.2 (Research & RAG Edition)
**Centro de Investigación Observatorio del Desarrollo (CIOdD) — Universidad de Costa Rica (UCR)**

Este notebook ejecuta el pipeline completo de estructuración documental con cero pérdida de texto (), extracción y clasificación de tablas, OCR adaptativo, generación de base DuckDB e índices TF-IDF y FAISS.

In [ ]:
# 1. Instalación de dependencias del sistema y Python
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-spa tesseract-ocr-eng
!pip -q install pymupdf pdfplumber pypdf pandas numpy pyarrow duckdb sentence-transformers faiss-cpu langdetect scikit-learn joblib tqdm pytesseract "pillow>=11.3,<12"
!mkdir -p /content/PDFs /content/pdf_intelligence
print("✅ Entorno preparado. Coloque sus archivos PDF en /content/PDFs")

In [ ]:
# 2. Clonar o importar el código del pipeline modular
!git clone https://github.com/tu-usuario/pdf-to-rag-pipeline.git /content/repo 2>/dev/null || true
import sys
sys.path.append("/content/repo/src")

from pdf_intelligence import PipelineConfig, CorpusBuilder, LocalSearch
print("✅ Módulo pdf_intelligence importado exitosamente")

In [ ]:
# 3. Ejecutar el pipeline sobre los PDFs
from pathlib import Path
config = PipelineConfig(
    pdf_directory=Path("/content/PDFs"),
    output_directory=Path("/content/pdf_intelligence"),
    ocr_enabled=True,
    chunk_target_words=550,
    chunk_overlap_words=80
)
builder = CorpusBuilder(config)
builder.process()

In [ ]:
# 4. Probar búsqueda local con TF-IDF
searcher = LocalSearch("/content/pdf_intelligence")
results = searcher.search_tfidf("inteligencia artificial", top_k=5)
for r in results:
    print(f"[{r["chunk_id"]}] Sec: {r["section_title"]} | Pág: {r["page_start"]}")
    print(r["text"][:200], "
")

In [ ]:
# 5. Descargar el corpus estructurado completo en ZIP
import shutil
shutil.make_archive("/content/pdf_intelligence_V52", "zip", "/content/pdf_intelligence")
try:
    from google.colab import files
    files.download("/content/pdf_intelligence_V52.zip")
except Exception as e:
    print("Descarga manual disponible en: /content/pdf_intelligence_V52.zip")